In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import PSNR,SSIM
import torch




# ── 核心函数 ───────────────────────────────────────────────────────────────────
def fft_lowpass_filter(img: np.ndarray, c: float):
    """
    对图像做 FFT，仅保留 100c% 的低频成分，高频置零，再逆变换。

    Parameters
    ----------
    img : np.ndarray
        输入图像，形状 (H, W) 灰度 或 (H, W, 3) RGB，dtype uint8。
    c   : float
        保留低频的比例，取值范围 (0, 1]。
        例如 c=0.1 表示只保留 10% 的低频，90% 高频置零。

    Returns
    -------
    filtered : np.ndarray
        经低通滤波后的图像，dtype uint8，形状与 img 相同。
    """
    assert 0 < c <= 1, "c 必须在 (0, 1] 之间"

    def _filter_channel(channel: np.ndarray) -> np.ndarray:
        H, W = channel.shape

        # 1. FFT & 中心化
        f_shift = np.fft.fftshift(np.fft.fft2(channel))

        # 2. 构造低通掩膜（保留中心 c*H × c*W 的矩形区域）
        mask = np.zeros((H, W), dtype=np.float32)
        rH = int(round(H * c / 2))   # 半径（行方向）
        rW = int(round(W * c / 2))   # 半径（列方向）
        cH, cW = H // 2, W // 2      # 频谱中心
        mask[cH - rH : cH + rH, cW - rW : cW + rW] = 1.0

        # 3. 应用掩膜，逆变换
        filtered_channel = np.fft.ifft2(np.fft.ifftshift(f_shift * mask))
        return np.clip(np.abs(filtered_channel), 0, 255).astype(np.uint8)

    # 灰度 or 彩色分别处理
    if img.ndim == 2:
        filtered = _filter_channel(img)
    else:
        filtered = np.stack([_filter_channel(img[:, :, ch]) for ch in range(img.shape[2])], axis=-1)

    return filtered


# ── 可视化 + 评价 ──────────────────────────────────────────────────────────────
def visualize_and_evaluate(img: np.ndarray, c: float):
    """
    调用 fft_lowpass_filter，展示三张子图并打印 PSNR / SSIM。

    子图布局：
        [原图]  [低通滤波图]  [差值图(原图-滤波图)]
    """
    filtered = fft_lowpass_filter(img, c)

    # 差值图（绝对值，拉伸到 0‑255 方便观察）
    diff = np.abs(img.astype(np.int16) - filtered.astype(np.int16)).astype(np.uint8)

    # ── 指标 ──
    def to_tensor(img_np):
        return torch.from_numpy(img_np).float() / 255.0

    # 你的 fft 函数输出的是 uint8 numpy，转一下再传给 PSNR/SSIM
    # img_tensor      = to_tensor(img_array)       # 原图
    filtered_tensor = to_tensor(filtered)        # fft_lowpass_filter 的输出

    # psnr_val = psnr(torch.tensor(img), filtered_tensor)
    # ssim_val = ssim(torch.tensor(img), filtered_tensor)
   
    print(f"c = {c:.2f}  ({c*100:.1f}%)")
    # print(f"  PSNR : {psnr_val:.4f} dB")
    # print(f"  SSIM : {ssim_val:.4f}")

    # ── 绘图 ──
    cmap = "gray" if img.ndim == 2 else None
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img,      cmap=cmap, vmin=0, vmax=255)
    axes[0].set_title("(Original)")
    axes[0].axis("off")

    axes[1].imshow(filtered, cmap=cmap, vmin=0, vmax=255)
    axes[1].set_title(f"(c={c:.2f})\n {c*100:.1f}% ")
    axes[1].axis("off")

    axes[2].imshow(diff,     cmap="hot", vmin=0, vmax=255)
    # axes[2].set_title(f"差值图 |原图 − 滤波图|\nPSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
    
    return to_tensor(img).permute(2,0,1), filtered_tensor.permute(2,0,1)

# ── 使用示例 ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    from PIL import Image
    
    pnsr = PSNR()
    ssim = SSIM()


    idx = 801
    pic = Image.open(f"/home/liuy/data/raw/DIV2K/DIV2K_valid_HR/{idx:04d}.png").convert("RGB")
    pic = np.array(pic)
    img, filter = visualize_and_evaluate(pic, 0.8)
    print(pnsr(img, filter), ssim(img, filter))
    # print(img[0,0,0], filter[0,0,0])

In [ ]:
from utils import *
import torch

res_data = torch.load("/home/liuy/work/projects/NeuralOperatorSR/ppft_study/res_data/res_tensor.pt",weights_only=True)

idx = 200

res = res_data[idx, 0]
fft = torch.fft.fft2(res)
fft = torch.fft.fftshift(fft)
pf_h, pf_v = ppft2(res)

Eng = torch.abs(fft)
Engpp = torch.abs(pf_h)
print(pf_h.shape)

plt.imshow(Eng, cmap='hot')
plt.imshow(Engpp, cmap='hot')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from utils.test_utils import fft_heatmap
from utils._utils import PSNR, SSIM

psnr = PSNR()
ssim = SSIM()
idx = 801
s = 2
patch_h_start = 0
patch_w_start = 0
patch_h = 512
patch_w = 600
# sheet = np.s_[:, patch_h_start:patch_h_start+patch_h, patch_w_start:patch_w_start+patch_w]
# sheet2 = np.s_[:, patch_h_start*s:patch_h_start*s+patch_h*s, patch_w_start*s:patch_w_start*s+patch_w*s]
gt_path = f"/home/liuy/data/raw/DIV2K/DIV2K_valid_HR/{idx:04d}.png"
lr_path = f"/home/liuy/data/raw/DIV2K/DIV2K_valid_LR_bicubic/X{s}/{idx:04d}x{s}.png"

gt = TF.to_tensor(TF.crop(Image.open(gt_path).convert("RGB"), top=patch_h_start*s, left=patch_w_start*s, height=patch_h*s, width=patch_w*s))
lr = TF.to_tensor(TF.crop(Image.open(lr_path).convert("RGB"), top=patch_h_start, left=patch_w_start, height=patch_h, width=patch_w)).unsqueeze(0)
# print(gt.shape, lr.shape)


bicubic = F.interpolate(lr, scale_factor=s, mode='bicubic', align_corners=False).squeeze(0)

_,_ = fft_heatmap(gt-bicubic, title="GT - Bicubic", )
print("Bicubic PSNR:", psnr(bicubic, gt))
print("Bicubic SSIM:", ssim(bicubic, gt))

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics.image import StructuralSimilarityIndexMeasure

class SSIM(nn.Module):
    def __init__(self, scale: int = 2, window_size: int = 11, sigma: float = 1.5):
        super().__init__()
        self.scale = scale
        self.window_size = window_size
        self.sigma = sigma
        # 先不注册buffer,在forward中根据通道数动态创建
        
    def _create_window(self, size: int, sigma: float) -> torch.Tensor:
        coords = torch.arange(size, dtype=torch.float32) - size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g /= g.sum()
        g_2d = g.unsqueeze(1) @ g.unsqueeze(0)
        return g_2d.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)

    def forward(self, sr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
        if sr.ndim == 3:
            sr = sr.unsqueeze(0)
            hr = hr.unsqueeze(0)

        sr_y = sr
        hr_y = hr
        
        # 获取通道数并创建对应的窗口
        num_channels = sr_y.shape[1]
        if not hasattr(self, '_window') or self._window.shape[0] != num_channels:
            window = self._create_window(self.window_size, self.sigma)
            # 复制到每个通道 (C, 1, H, W)
            self._window = window.repeat(num_channels, 1, 1, 1).to(sr_y.device)
        
        c1 = (0.01) ** 2
        c2 = (0.03) ** 2
        pad = self.window_size // 2

        # 使用 groups=num_channels 确保每个通道独立计算
        mu1 = F.conv2d(sr_y, self._window, padding=pad, groups=num_channels)
        mu2 = F.conv2d(hr_y, self._window, padding=pad, groups=num_channels)
        
        mu1_sq = mu1 ** 2
        mu2_sq = mu2 ** 2
        mu1mu2 = mu1 * mu2

        sigma1_sq = F.conv2d(sr_y * sr_y, self._window, padding=pad, groups=num_channels) - mu1_sq
        sigma2_sq = F.conv2d(hr_y * hr_y, self._window, padding=pad, groups=num_channels) - mu2_sq
        sigma12 = F.conv2d(sr_y * hr_y, self._window, padding=pad, groups=num_channels) - mu1mu2

        num = (2 * mu1mu2 + c1) * (2 * sigma12 + c2)
        den = (mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2)
        
        ssim_map = num / den  # (B, C, H, W)
        
        # 先对空间维度平均,再对通道平均,最后对batch平均
        return ssim_map.mean(dim=[2, 3]).mean(dim=1).mean(dim=0)
    
def rgb_to_y(img: torch.Tensor) -> torch.Tensor:
    """
    RGB -> Y (亮度通道), ITU-R BT.601
    img: (B, 3, H, W) or (3, H, W), range [0, 1]
    return: (B, 1, H, W) or (1, H, W), range [16/255, 235/255]
    """
    if img.ndim == 3:
        img = img.unsqueeze(0)
        squeeze = True
    else:
        squeeze = False

    r, g, b = img[:, 0:1], img[:, 1:2], img[:, 2:3]
    y = 16.0/255.0 + (65.481/255.0)*r + (128.553/255.0)*g + (24.966/255.0)*b

    return y.squeeze(0) if squeeze else y

# 1. 假设这是你自己写的 ssim_fn (示例中使用一个简单的占位逻辑)
# 请将这里替换为你实际的 ssim_fn
ssim_fn = SSIM()

# 2. 模拟数据准备
torch.manual_seed(42)
num_samples = 10
# 随机生成 10 张 64x64 的单通道图片 (模拟 [N, C, H, W])
preds = torch.rand((num_samples, 3, 64, 64))
refs = torch.rand((num_samples, 3, 64, 64))

# ---------------------------------------------------------
# 方法 A: 使用 PyTorch Lightning (torchmetrics) 的计算方式
# 确保torchmetrics也是单通道计算
pl_ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0)
pl_result = pl_ssim_metric(preds, refs)  # preds/refs: [10, 1, 64, 64]

# ---------------------------------------------------------
# 方法 B: 使用你的 ssim_fn 逐张计算并取平均
# ---------------------------------------------------------
individual_scores = []
for i in range(num_samples):
    # 提取单张图片 [C, H, W]
    s_pred = preds[i]
    s_ref = refs[i]
    
    # 逐张计算
    score = ssim_fn(s_pred, s_ref)
    individual_scores.append(score)

# 手动求平均
your_result = torch.stack(individual_scores).mean()

# 3. 结果对比
print(f"样本数量: {num_samples}")
print(f"PL (torchmetrics) 的平均 SSIM: {pl_result.item():.6f}")
print(f"你的 ssim_fn 逐张求和后的平均值: {your_result.item():.6f}")
print(f"二者差异: {torch.abs(pl_result - your_result).item():.6e}")

样本数量: 10
PL (torchmetrics) 的平均 SSIM: 0.003031
你的 ssim_fn 逐张求和后的平均值: 0.063114
二者差异: 6.008244e-02
